# 01a — Load and clean 3-D hologram stacks

Load one detector acquisition without collapsing its frame axis, subtract an averaged and optionally linearly fitted dark, inspect its intensities, threshold the corrected frames, average them, and save one compact 2-D image using its acquisition ID.

In [1]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import erf
import matplotlib.pyplot as plt


def fit_horizontal_band(
    pattern,
    nedge=10,nstart=0,
    band_center=None,
    band_width=None,
    band_edge=None,
    band_amplitude=None,
    bump_center=None,
    bump_sigma=None,
    plot=True,
):
    """
    Fit a horizontal detector-band artifact using the left/right edges
    of a 2D scattering pattern.

    Model
    -----
    measured profile =
        polynomial background
        + Gaussian bump
        + negative smoothed-box artifact

    Parameters
    ----------
    pattern : 2D ndarray
        Scattering pattern.

    nedge : int
        Number of columns averaged on each side.

    band_center : float, optional
        Initial estimate of band center [pixels].
        Default = image center.

    band_width : float, optional
        Initial estimate of band width [pixels].

    band_edge : float, optional
        Initial estimate of edge smoothing [pixels].

    band_amplitude : float, optional
        Initial estimate of the depth of the negative band.

    bump_center : float, optional
        Initial estimate of the Gaussian bump center [pixels].

    bump_sigma : float, optional
        Initial estimate of Gaussian sigma [pixels].

    plot : bool
        If True, show diagnostic plots.

    Returns
    -------
    band_2d : ndarray
        Estimated NEGATIVE horizontal-band artifact,
        same shape as pattern.

        Correct with:

            corrected = pattern - band_2d

    gaussian_2d : ndarray
        Fitted Gaussian contribution to the natural profile,
        repeated horizontally to match pattern.shape.

    fit_info : dict
        Fitted parameters and 1D fit components.
    """

    pattern = np.asarray(pattern, dtype=float)

    if pattern.ndim != 2:
        raise ValueError("pattern must be a 2D array.")

    ny, nx = pattern.shape
    y = np.arange(ny, dtype=float)

    # =========================================================
    # Get vertical profiles from left/right detector edges
    # =========================================================

    left_profile = np.nanmean(pattern[:, nstart:nedge+nstart], axis=1)
    right_profile = np.nanmean(pattern[:, -nedge-nstart:-nstart], axis=1)

    profile = 0.5 * (left_profile + right_profile)

    # =========================================================
    # Model components
    # =========================================================

    def smooth_box(y, amplitude, center, width, edge_sigma):
        """
        Positive smooth box made from two error-function edges.
        """

        y1 = center - width / 2
        y2 = center + width / 2

        return amplitude / 2 * (
            erf((y - y1) / (np.sqrt(2) * edge_sigma))
            - erf((y - y2) / (np.sqrt(2) * edge_sigma))
        )

    def polynomial_background(y, c0, c1, c2):
        """
        Slowly varying polynomial background.
        """

        x = y - ny / 2

        return c0 + c1 * x + c2 * x**2

    def gaussian_bump(y, amplitude, center, sigma):
        """
        Broad Gaussian scattering contribution.
        """

        return amplitude * np.exp(
            -0.5 * ((y - center) / sigma)**2
        )

    def model(
        y,
        c0,
        c1,
        c2,
        bump_amp,
        bump_center_fit,
        bump_sigma_fit,
        band_amp,
        band_center_fit,
        band_width_fit,
        band_edge_fit,
    ):

        polynomial = polynomial_background(
            y,
            c0,
            c1,
            c2,
        )

        gaussian = gaussian_bump(
            y,
            bump_amp,
            bump_center_fit,
            bump_sigma_fit,
        )

        band = smooth_box(
            y,
            band_amp,
            band_center_fit,
            band_width_fit,
            band_edge_fit,
        )

        return polynomial + gaussian - band

    # =========================================================
    # Initial guesses
    # =========================================================

    if band_center is None:
        band_center = ny / 2

    if band_width is None:
        band_width = ny * 0.08

    if band_edge is None:
        band_edge = max(2, band_width / 10)

    if bump_center is None:
        bump_center = ny / 2

    if bump_sigma is None:
        bump_sigma = ny / 3

    baseline = np.nanmedian(profile)

    bump_amplitude = max(
        np.nanmax(profile) - baseline,
        1e-12
    )

    # ---------------------------------------------------------
    # Estimate initial band depth if not supplied
    # ---------------------------------------------------------

    if band_amplitude is None:

        distance = np.abs(y - band_center)

        inside = distance < band_width / 2

        outside = (
            (distance > band_width)
            & (distance < 2 * band_width)
        )

        if np.any(inside) and np.any(outside):

            band_amplitude = (
                np.nanmedian(profile[outside])
                - np.nanmedian(profile[inside])
            )

        else:

            band_amplitude = 0.05 * (
                np.nanmax(profile) - np.nanmin(profile)
            )

        band_amplitude = max(
            band_amplitude,
            1e-12
        )

    p0 = [
        baseline,          # c0
        0.0,               # c1
        0.0,               # c2

        bump_amplitude,
        bump_center,
        bump_sigma,

        band_amplitude,
        band_center,
        band_width,
        band_edge,
    ]

    # =========================================================
    # Fit bounds
    # =========================================================

    lower = [
        -np.inf,       # c0
        -np.inf,       # c1
        -np.inf,       # c2

        0,             # bump amplitude
        0,             # bump center
        1,             # bump sigma

        0,             # band amplitude
        0,             # band center
        1,             # band width
        0.2,           # band edge
    ]

    upper = [
        np.inf,
        np.inf,
        np.inf,

        np.inf,
        ny,
        2 * ny,

        np.inf,
        ny,
        ny,
        ny / 2,
    ]

    # =========================================================
    # Fit
    # =========================================================

    valid = np.isfinite(profile)

    popt, pcov = curve_fit(
        model,
        y[valid],
        profile[valid],
        p0=p0,
        bounds=(lower, upper),
        maxfev=50000,
    )

    # =========================================================
    # Separate fitted components
    # =========================================================

    fitted_polynomial = polynomial_background(
        y,
        *popt[:3]
    )

    fitted_gaussian = gaussian_bump(
        y,
        *popt[3:6]
    )

    fitted_band_positive = smooth_box(
        y,
        *popt[6:]
    )

    # Actual detector artifact is negative
    band_1d = -fitted_band_positive

    # Natural profile without artifact
    fitted_background = (
        fitted_polynomial
        + fitted_gaussian
    )

    # Complete measured-profile model
    fitted_total = (
        fitted_polynomial
        + fitted_gaussian
        + band_1d
    )

    # =========================================================
    # Convert 1D components to 2D
    # =========================================================

    band_2d = np.repeat(
        band_1d[:, None],
        nx,
        axis=1
    )

    gaussian_2d = np.repeat(
        fitted_gaussian[:, None],
        nx,
        axis=1
    )

    # =========================================================
    # Diagnostic plots
    # =========================================================

    if plot:

        # -----------------------------------------------------
        # Complete fit
        # -----------------------------------------------------

        plt.figure(figsize=(10, 5))

        plt.plot(
            y,
            left_profile,
            alpha=0.4,
            label="Left edge"
        )

        plt.plot(
            y,
            right_profile,
            alpha=0.4,
            label="Right edge"
        )

        plt.plot(
            y,
            profile,
            linewidth=1.5,
            label="Average edge profile"
        )

        plt.plot(
            y,
            fitted_total,
            linewidth=2.5,
            label="Complete fit"
        )

        plt.plot(
            y,
            fitted_background,
            "--",
            linewidth=2,
            label="Natural profile"
        )

        plt.plot(
            y,
            fitted_polynomial,
            "--",
            alpha=0.7,
            label="Polynomial background"
        )

        plt.plot(
            y,
            fitted_polynomial + fitted_gaussian,
            linewidth=1.5,
            label="Polynomial + Gaussian"
        )

        plt.fill_between(
            y,
            fitted_background,
            fitted_total,
            alpha=0.2,
            label="Band artifact"
        )

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Horizontal-band fit")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Individual extracted components
        # -----------------------------------------------------

        plt.figure(figsize=(10, 4))

        plt.plot(
            y,
            fitted_gaussian,
            label="Gaussian bump"
        )

        plt.plot(
            y,
            band_1d,
            label="Band artifact"
        )

        plt.axhline(
            0,
            linewidth=1
        )

        plt.xlabel("Detector row [pixel]")
        plt.ylabel("Intensity")
        plt.title("Extracted fit components")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # -----------------------------------------------------
        # Parameters
        # -----------------------------------------------------

        print("\nFitted Gaussian:")
        print(f"  amplitude = {popt[3]:.6g}")
        print(f"  center    = {popt[4]:.2f} px")
        print(f"  sigma     = {popt[5]:.2f} px")

        print("\nFitted horizontal band:")
        print(f"  amplitude = {popt[6]:.6g}")
        print(f"  center    = {popt[7]:.2f} px")
        print(f"  width     = {popt[8]:.2f} px")
        print(f"  edge sigma= {popt[9]:.2f} px")

    # =========================================================
    # Output information
    # =========================================================

    names = [
        "background_offset",
        "background_linear",
        "background_quadratic",
        "bump_amplitude",
        "bump_center",
        "bump_sigma",
        "band_amplitude",
        "band_center",
        "band_width",
        "band_edge",
    ]

    fit_info = {
        "parameters": dict(zip(names, popt)),
        "covariance": pcov,

        "left_profile": left_profile,
        "right_profile": right_profile,
        "profile": profile,

        "polynomial": fitted_polynomial,
        "gaussian_1d": fitted_gaussian,
        "background": fitted_background,

        "band_1d": band_1d,

        "fit_profile": fitted_total,
    }

    return band_2d, gaussian_2d, fit_info
    
    
    
    
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector

def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())
    
BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)

from data_loading import SextantsNexusLoader, load_processing
print("Base folder:", BASEFOLDER)


import os, sys
from os.path import join
from getpass import getuser
from importlib import reload

import h5py
import numpy as np
import matplotlib.pyplot as plt
import skimage.morphology
from pyFAI.detectors import Detector


def find_basefolder(start=None):
    return os.path.abspath(start or os.getcwd())


BASEFOLDER = find_basefolder()
sys.path.append(join(BASEFOLDER, "library"))
print("Base folder:", BASEFOLDER)
import fthcore as fth
import helper_functions as helper
import interactive
from interactive import cimshow
import mask_lib
import reconstruct_rb as rec
import PETRA_MaxP04_loading as loading
import fth_phase_workflow as wf

%matplotlib qt

/usr/lib/python3/dist-packages/pytools/persistent_dict.py:52: RecommendedHashNotFoundWarning: Unable to import recommended hash 'siphash24.siphash13', falling back to 'hashlib.sha256'. Run 'python3 -m pip install siphash24' to install the recommended hash.
  warn("Unable to import recommended hash 'siphash24.siphash13', "


Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW
Base folder: /home/experiences/sextants/com-sextants/SEXT_NEW


## Configuration

In [2]:

RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"

IMAGE_ID = 569
DARK_IDS = [574]


## Load the 3-D arrays

In [3]:
loader = SextantsNexusLoader(RAW_FOLDER)

# load_processing returns the simple 2-D average and the full 3-D stack.
blind_average, stack = load_processing(loader, IMAGE_ID)

# The dark is also a 3-D acquisition. Use its simple frame average.
dark_reference, dark_stack = load_processing(loader, DARK_IDS)
dark_reference = np.asarray(dark_reference, dtype=float)
print(f"Loaded ID {IMAGE_ID}: stack {stack.shape}, average {blind_average.shape}")
print("Dark stack:", dark_stack.shape, "dark average:", dark_reference.shape)

Loaded ID 569: stack (101, 2048, 2048), average (2048, 2048)
Dark stack: (11, 2048, 2048) dark average: (2048, 2048)


## Dark subtraction

Fit `frame ≈ scale × dark + offset` on low-intensity pixels from a small selected detector region. The fitted dark background is then subtracted from every pixel of the complete frame.

In [4]:
plt.close("all")


# Dark-rescaling controls are here because they apply to this operation.
FIT_DARK_LINEAR = True
DARK_FIT_PERCENTILE = 100
DARK_FIT_ROWS = slice(0, 100)
DARK_FIT_COLUMNS = slice(0, 100)
DARK_FIT_STRIDE = 1  # Optional subsampling within the selected region.

mask_detector=1.*(plt.imread("processed/mask_pixels/mask_detector.png")[:,:,0]==1)


def subtract_fitted_dark(frame, dark, rows, columns, percentile=30, stride=1):
    # Estimate scale and offset only from the selected detector region.
    sample_image = frame[rows, columns][::stride, ::stride].ravel().astype(float)
    sample_dark = dark[rows, columns][::stride, ::stride].ravel().astype(float)
    valid = np.isfinite(sample_image) & np.isfinite(sample_dark)
    limit = np.percentile(sample_image[valid], percentile)
    valid &= sample_image <= limit
    if valid.sum() < 2 or np.ptp(sample_dark[valid]) == 0:
        scale, offset = 1.0, 0.0
    else:
        scale, offset = np.polyfit(sample_dark[valid], sample_image[valid], 1)
    corrected = frame - (scale * dark + offset)
    return corrected.astype(np.float32), float(scale), float(offset)

corrected_frames = []
dark_fits = []
for frame in stack:
    if FIT_DARK_LINEAR:
        corrected, scale, offset = subtract_fitted_dark(
            frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
            DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
        )
    else:
        corrected, scale, offset = frame - dark_reference, 1.0, 0.0
    corrected_frames.append(corrected)
    dark_fits.append((scale, offset))
corrected_stack = np.stack(corrected_frames)
dark_fits = np.asarray(dark_fits)
print("Dark scale mean:", dark_fits[:, 0].mean(),
      "offset mean:", dark_fits[:, 1].mean())

Dark scale mean: 0.6998436288481819 offset mean: 53.68117522733608


In [7]:
cimshow(np.clip(np.average(corrected_stack, axis=0), None, 99.0))

interactive(children=(FloatRangeSlider(value=(-14.048226464271545, 99.0), description='contrast', layout=Layou…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [8]:
cimshow(np.clip(corrected_stack[0], None, 99.0))

interactive(children=(FloatRangeSlider(value=(-29.03529167175293, 99.0), description='contrast', layout=Layout…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [9]:
# calculate and subtract the band

cimshow(np.clip(np.average(corrected_stack, axis=0), None, 99.0))

interactive(children=(FloatRangeSlider(value=(-14.048226464271545, 99.0), description='contrast', layout=Layou…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [10]:
plt.close("all")

band_do=False
nedge=100
nstart=50

if band_do:
    band, gaussian,fit = fit_horizontal_band(
        np.average(corrected_stack, axis=0),
        nedge=nedge,nstart=nstart,
        band_center=1024,
        band_width=80,
        band_edge=11,
    )

In [11]:
plt.close("all")
if band_do:
    cimshow(np.clip(np.average(corrected_stack, axis=0)-band-gaussian, None, 99.0))

In [12]:
if band_do:
    corrected_stack=corrected_stack-band-gaussian

## Inspect the dark rescaling fit

The scatter plot uses the first frame of the selected image. Grey points are all finite sampled pixels, blue points are the pixels used for the linear fit, and the red line is the fitted dark background.

In [13]:
# Show the same pixels and selection used for the first frame's fit.
image_values = stack[0, DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
dark_values = dark_reference[DARK_FIT_ROWS, DARK_FIT_COLUMNS][
    ::DARK_FIT_STRIDE, ::DARK_FIT_STRIDE
].ravel().astype(float)
finite = np.isfinite(image_values) & np.isfinite(dark_values)
fit_limit = np.percentile(image_values[finite], DARK_FIT_PERCENTILE)
used = finite & (image_values <= fit_limit)
scale, offset = dark_fits[0]

fig, axis = plt.subplots(figsize=(6, 5))
axis.scatter(dark_values[finite], image_values[finite], s=3, alpha=0.08,
             color="0.4", rasterized=True, label="sampled pixels")
axis.scatter(dark_values[used], image_values[used], s=4, alpha=0.25,
             color="tab:blue", rasterized=True, label="pixels used for fit")
x_line = np.linspace(dark_values[used].min(), dark_values[used].max(), 200)
axis.plot(x_line, scale * x_line + offset, color="red", linewidth=2,
          label=f"fit: y = {scale:.4g} x + {offset:.4g}")
axis.set_title(f"ID {IMAGE_ID}: first-frame dark fit")
axis.set_xlabel("Dark-reference intensity")
axis.set_ylabel("Raw-frame intensity")
axis.legend()
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Inspect corrected intensities

Choose the thresholds in the next cell, then rerun that cell and the cleaning cell below.

In [14]:

plt.close("all")


# Edit these values while inspecting the histograms below.
INTENSITY_THRESHOLD = 20.0
HISTOGRAM_RANGE = (-20, 80)
HISTOGRAM_BINS = 400
PHOTON_VIEW_ROWS = slice(402, 900)
PHOTON_VIEW_COLUMNS = slice(420, 900)
gauss=False


representative_image = corrected_stack[0]
fig, axis = plt.subplots(figsize=(6, 4))
axis.hist(representative_image.ravel(), bins=HISTOGRAM_BINS, range=HISTOGRAM_RANGE, histtype="step")
axis.axvline(INTENSITY_THRESHOLD, color="red", linestyle="--", label="threshold")
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Intensity")
axis.set_ylabel("Pixel count")
axis.set_xlim(*HISTOGRAM_RANGE)
axis.legend()
axis.set_yscale("log")
plt.tight_layout()
plt.show()

import scipy
def gauss(image, sigma=3):
    return scipy.ndimage.gaussian_filter(image, sigma=sigma)
    
# Inspect the same first frames on the scale of individual photon events.
fig, axis = plt.subplots(figsize=(6, 5))
photon_view = corrected_stack[0, PHOTON_VIEW_ROWS, PHOTON_VIEW_COLUMNS]
if gauss:
    image = axis.imshow(
    photon_view*(gauss(photon_view, 2)>(INTENSITY_THRESHOLD/1.5)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
else:
    image = axis.imshow(
    photon_view*((photon_view)>(INTENSITY_THRESHOLD)),
        vmin=HISTOGRAM_RANGE[0], vmax=HISTOGRAM_RANGE[1], cmap="viridis")
    
axis.set_title(f"ID {IMAGE_ID}: first dark-corrected frame")
axis.set_xlabel("Column in selected region")
axis.set_ylabel("Row in selected region")
fig.colorbar(image, ax=axis, label="Corrected intensity")
plt.tight_layout()
plt.show()

## Threshold, clip, and average

In [15]:
plt.close("all")


In [16]:
# Subtract the chosen threshold from every frame, clip negatives, then average.
if gauss:
    blurred_stack=corrected_stack.copy()
    for i in range(blurred_stack.shape[0]):
        blurred_stack[i]=corrected_stack[i]*(gauss(corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
    cleaned_stack = np.clip(blurred_stack - 0*float(INTENSITY_THRESHOLD), 0, None)
else:
    blurred_stack=corrected_stack.copy()
    cleaned_stack = np.clip(blurred_stack - float(INTENSITY_THRESHOLD), 0, None)


cleaned_average = np.mean(cleaned_stack, axis=0, dtype=np.float64)
print("Threshold:", INTENSITY_THRESHOLD, "average shape:", cleaned_average.shape)

# Compare the blind average returned by load_processing with the cleaned average.
# Each column uses one shared linear color scale so the change is directly visible.
AVERAGE_DISPLAY_PERCENTILES = (1, 10.9)

comparison_values = np.concatenate((blind_average.ravel(), cleaned_average.ravel()))
comparison_values = comparison_values[np.isfinite(comparison_values)]
vmin, vmax = np.percentile(comparison_values, AVERAGE_DISPLAY_PERCENTILES)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))


for axis, image, description in zip(
    axes,
    (blind_average, cleaned_average),
    ("blind load_processing average", "cleaned average"),
):
    shown = axis.imshow(image, vmin=vmin, vmax=vmax, cmap="viridis")
    axis.set_title(f"ID {IMAGE_ID}: {description} (linear scale)")
    axis.set_axis_off()
    fig.colorbar(shown, ax=axis, label="Average intensity")
plt.tight_layout()
plt.show()

Threshold: 20.0 average shape: (2048, 2048)


In [17]:
temp=np.clip(cleaned_average, None, 1000)
cimshow(temp)

interactive(children=(FloatRangeSlider(value=(0.0, 1000.0), description='contrast', layout=Layout(width='500px…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [18]:
cimshow(np.abs(fth.reconstruct(cleaned_average)))

interactive(children=(FloatRangeSlider(value=(0.00033248912081006393, 4.275983917019852), description='contras…

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [19]:
temp=np.clip(blind_average, None, 10000)
cimshow(temp)

interactive(children=(FloatRangeSlider(value=(165.16831683168317, 9701.878811881683), description='contrast', …

(<Figure size 700x700 with 1 Axes>, <Axes: >)

In [20]:
BASEFOLDER="/home/experiences/sextants/com-sextants/SEXT_NEW"

## Save one cleaned average per acquisition

In [21]:
plt.close("all")

OUTPUT_FOLDER = BASEFOLDER + "/processed/" + "cleaned_acquisitions"
USER = "rb"

setup_metadata = loader.load(IMAGE_ID).metadata
output_file = OUTPUT_FOLDER + f"cleaned_ImId_{IMAGE_ID:04d}_{USER}.npz"
np.savez_compressed(
    output_file,
    image=cleaned_average,
    blind_average=blind_average,
    image_id=np.asarray(IMAGE_ID, dtype=int),
    dark_ids=np.asarray(DARK_IDS, dtype=int),
    threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
    dark_fits=dark_fits,
    energy_eV=float(setup_metadata["energy_eV"]),
    ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
    px_size_m=11.0e-6,
)
print(f"Saved ID {IMAGE_ID}: {output_file}")

Saved ID 569: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitionscleaned_ImId_0569_rb.npz


In [23]:
plt.close("all")

# After tuning the parameters above, put the remaining image IDs here.
# This repeats loading, dark subtraction, thresholding, averaging, and saving
# without producing diagnostic plots. An empty list does nothing.
BATCH_IMAGE_IDS = list(np.arange(575,584) )

for batch_image_id in BATCH_IMAGE_IDS:
    batch_blind_average, batch_stack = load_processing(loader, batch_image_id)

    batch_corrected_frames = []
    batch_dark_fits = []
    for frame in batch_stack:
        if FIT_DARK_LINEAR:
            corrected, scale, offset = subtract_fitted_dark(
                frame, dark_reference, DARK_FIT_ROWS, DARK_FIT_COLUMNS,
                DARK_FIT_PERCENTILE, DARK_FIT_STRIDE
            )
        else:
            corrected, scale, offset = frame - dark_reference, 1.0, 0.0
        batch_corrected_frames.append(corrected)
        batch_dark_fits.append((scale, offset))

    batch_corrected_stack = np.stack(batch_corrected_frames)


    if band_do:
        band, gaussian,fit = fit_horizontal_band(
        np.average(batch_corrected_stack, axis=0),
        nedge=nedge,nstart=nstart,
        band_center=1024,
        band_width=80,
        band_edge=11,
        )
        batch_corrected_stack=batch_corrected_stack-band-gaussian

    if gauss:
        blurred_stack=batch_corrected_stack.copy()
        for i in range(blurred_stack.shape[0]):
            blurred_stack[i]=batch_corrected_stack[i]*(gauss(batch_corrected_stack[i], 2)>(INTENSITY_THRESHOLD/2))
        batch_cleaned_stack = np.clip(blurred_stack- 0*float(INTENSITY_THRESHOLD), 0, None)
    else:
        blurred_stack=batch_corrected_stack.copy()
        batch_cleaned_stack = np.clip(blurred_stack- float(INTENSITY_THRESHOLD), 0, None)

    batch_cleaned_average = np.mean(
        batch_cleaned_stack, axis=0, dtype=np.float64
    )
    batch_output_file = (
        OUTPUT_FOLDER + f"/cleaned_ImId_{batch_image_id:04d}_{USER}.npz"
    )
    np.savez_compressed(
        batch_output_file,
        image=batch_cleaned_average,
        blind_average=batch_blind_average,
        image_id=np.asarray(batch_image_id, dtype=int),
        dark_ids=np.asarray(DARK_IDS, dtype=int),
        threshold=np.asarray(INTENSITY_THRESHOLD, dtype=float),
        dark_fits=np.asarray(batch_dark_fits),
        energy_eV=float(setup_metadata["energy_eV"]),
        ccd_dist_m=float(setup_metadata["ccd_dist_m"]),
        px_size_m=11.0e-6,
    )
    print(f"Saved ID {batch_image_id}: {batch_output_file}")

print("fine-tuned im_id:", IMAGE_ID)
print("batch im_ids:", BATCH_IMAGE_IDS)
print("dark_ids:", DARK_IDS)

Saved ID 575: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0575_rb.npz
Saved ID 576: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0576_rb.npz
Saved ID 577: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0577_rb.npz
Saved ID 578: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0578_rb.npz
Saved ID 579: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0579_rb.npz
Saved ID 580: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0580_rb.npz
Saved ID 581: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0581_rb.npz
Saved ID 582: /home/experiences/sextants/com-sextants/SEXT_NEW/processed/cleaned_acquisitions/cleaned_ImId_0582_rb.npz


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/scanx_0583.nxs', errno = 2, error message = 'Aucun fichier ou dossier de ce nom', flags = 0, o_flags = 0)